In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
#pip install ipywidgets ## install the library
import ipywidgets as widgets
import os, glob


In [2]:

def Export_folders_from_results(PATH):
    """Detect all subfolders inside PATH (simulation results)."""
    simulation_dirs = sorted([
        os.path.join(PATH, d)
        for d in os.listdir(PATH)
        if os.path.isdir(os.path.join(PATH, d))
    ])
    for s in simulation_dirs:
        print(s)
    return simulation_dirs  # opcional, útil si quieres usarlas después

def create_quad_viewer(png_directories, simulation_names=None):
    """
    Flexible viewer: shows any number of simulations (3, 4, 5, ...)
    in a grid layout with individual and synchronized controls.
    """

    all_png_files = []
    all_widgets = []
    all_sliders = []
    all_labels = []
    
    # Load images for each simulation
    for i, png_dir in enumerate(png_directories):
        png_files = sorted(glob.glob(os.path.join(png_dir, "*.png")))
        print(f"Found {len(png_files)} PNG files in {png_dir}")
        all_png_files.append(png_files)
        
        image_widget = widgets.Image(
            value=open(png_files[0], "rb").read(),
            format='png',
            width=400,
            height=300
        )
        all_widgets.append(image_widget)
        
        slider = widgets.IntSlider(
            value=0,
            min=0,
            max=len(png_files)-1,
            step=1,
            description=f'{simulation_names[i]}:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='400px')
        )
        all_sliders.append(slider)
        
        label = widgets.Label(
            value=f"Timestep 0/{len(png_files)-1}: {os.path.basename(png_files[0])}",
            layout=widgets.Layout(width='400px'),   
        )
        all_labels.append(label)
        
        # Update function (closure)
        def make_update_function(idx):
            def update_image(change):
                frame_idx = change['new']
                files = all_png_files[idx]
                with open(files[frame_idx], "rb") as f:
                    all_widgets[idx].value = f.read()
                all_labels[idx].value = f"Timestep {frame_idx}/{len(files)-1}: {os.path.basename(files[frame_idx])}"
            return update_image
        slider.observe(make_update_function(i), names='value')
    
    # Master control slider
    master_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=min([len(files) for files in all_png_files])-1,
        step=1,
        description='Sync All:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='100%')
    )

    def sync_all(change):
        frame_idx = change['new']
        for slider in all_sliders:
            if frame_idx <= slider.max:
                slider.value = frame_idx
    master_slider.observe(sync_all, names='value')
    
    rows = []
    for i in range(0, len(all_widgets), 2):
        imgs = all_widgets[i:i+2]
        lbls = all_labels[i:i+2]
        slds = all_sliders[i:i+2]
        rows.append(widgets.HBox(lbls,layout=widgets.Layout(justify_content='center'),    width='100%',height='auto'))
        rows.append(widgets.HBox(imgs,layout=widgets.Layout(justify_content='center'),   width='100%', height='auto'))
        rows.append(widgets.HBox(slds,layout=widgets.Layout(justify_content='center'),   width='100%', height='auto'))

    viewer = widgets.VBox([
        widgets.HTML("<h3>Timesteps</h3>"),
        master_slider,
        widgets.HTML("<h3>Simulations</h3>"),
        *rows
    ])
    
    return viewer
 
 
 
from PIL import Image
import matplotlib.pyplot as plt
def plot_timesteps_images(folder, start, end, step=10, ncols=3, figsize=(12, 6), pad=3):
    """
    Displays a grid of PNG images for selected timesteps (with zero-padded filenames).

    Parameters
    ----------
    folder : str
        Directory where images are stored (named like 000.png, 001.png, ...).
    start : int
        First timestep to display.
    end : int
        Last timestep to display.
    step : int, optional
        Step interval between images (default = 10).
    ncols : int, optional
        Number of columns in the plot grid.
    figsize : tuple, optional
        Figure size for matplotlib.
    pad : int, optional
        Number of zero-padding digits (e.g., 3 → 000, 001, 020).
    """
    timesteps = list(range(start, end + 1, step))
    nrows = (len(timesteps) + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten()
    
    for i, t in enumerate(timesteps):
        filename = os.path.join(folder, f"{t:0{pad}d}.png")
        if os.path.exists(filename):
            img = Image.open(filename)
            axes[i].imshow(img)
            axes[i].set_title(f"Timestep {t}")
            axes[i].axis("off")
        else:
            axes[i].text(0.5, 0.5, f"Missing\n{t:0{pad}d}.png", ha='center', va='center')
            axes[i].axis("off")
    
    # Hide extra axes
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    
    plt.tight_layout()
    plt.show()

In [3]:
PATH = "/scratch/flore0a/Preproposal/Scalability/Test1"  # Folder where all sims are located
Directorio_results = Export_folders_from_results(PATH)
## add "/Simulation2D" in the Directorio_results list for each simulation
Directorio_results = [d + "/Simulation2D" for d in Directorio_results]
Directorio_results = Directorio_results[1:] 


/scratch/flore0a/Preproposal/Scalability/Test1/TimingResults
/scratch/flore0a/Preproposal/Scalability/Test1/procesors192
/scratch/flore0a/Preproposal/Scalability/Test1/procesors384
/scratch/flore0a/Preproposal/Scalability/Test1/procesors450
/scratch/flore0a/Preproposal/Scalability/Test1/procesors48
/scratch/flore0a/Preproposal/Scalability/Test1/procesors576
/scratch/flore0a/Preproposal/Scalability/Test1/procesors96


In [4]:
#how to remove the previos before /
print(Directorio_results)

name = [ os.path.basename(os.path.dirname(path)) for path in Directorio_results]
png_directory = [d + "/png" for d in Directorio_results] # add the /png folder
print(png_directory)

quad_viewer = create_quad_viewer(png_directory, name)
display(quad_viewer)

['/scratch/flore0a/Preproposal/Scalability/Test1/procesors192/Simulation2D', '/scratch/flore0a/Preproposal/Scalability/Test1/procesors384/Simulation2D', '/scratch/flore0a/Preproposal/Scalability/Test1/procesors450/Simulation2D', '/scratch/flore0a/Preproposal/Scalability/Test1/procesors48/Simulation2D', '/scratch/flore0a/Preproposal/Scalability/Test1/procesors576/Simulation2D', '/scratch/flore0a/Preproposal/Scalability/Test1/procesors96/Simulation2D']
['/scratch/flore0a/Preproposal/Scalability/Test1/procesors192/Simulation2D/png', '/scratch/flore0a/Preproposal/Scalability/Test1/procesors384/Simulation2D/png', '/scratch/flore0a/Preproposal/Scalability/Test1/procesors450/Simulation2D/png', '/scratch/flore0a/Preproposal/Scalability/Test1/procesors48/Simulation2D/png', '/scratch/flore0a/Preproposal/Scalability/Test1/procesors576/Simulation2D/png', '/scratch/flore0a/Preproposal/Scalability/Test1/procesors96/Simulation2D/png']
Found 25 PNG files in /scratch/flore0a/Preproposal/Scalability/Tes